> This notebook requires private benchmark data files that are not included in the public repository. See `../data/README.md` and `../labels/README.md`.


# Healthy Baseline EDA - Vibration RMS Features

This notebook explores the vibration dataset for the private sensor benchmark using the preprocessing contract adopted in this simplified production version of the pipeline.

The analysis is organized around two core concepts:

1. **Self healthy baseline**  
   Each scenario is first treated as its own asset, and the `.fit` portion is used to define what healthy behavior looks like for that scenario.

2. **Degradation analysis and grouping surrogate**  
   Because the dataset is anonymous and does not include the machine metadata that would normally define asset families, we use observed asset/time-series behavior as a surrogate for grouping during offline analysis. In a real production setting, this grouping would come from known metadata such as machine type, operating context, or sensor location, not from retrospective signal inspection.

   Different alarm parameters are used for each group, but the model is the same for all.

**Analysis stages:**
1. Dataset loading and scenario/label inspection
2. Preprocessing contract - spike clipping plus trusted `uptime`
3. Healthy baseline definition - raw/scaled RMS behavior and residual calibration
4. Degradation inspection across scenarios
5. Manual grouping of scenarios as a surrogate for missing asset metadata

> Model evaluation and alert engine debugging are covered in `02_model_debugging.ipynb`.


In [ ]:
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

In [ ]:
from analysis.plotting import (
    create_group_distribution_widget,
    create_rms_scenario_widget,
    create_scenario_inspector,
    create_sigmoid_global_residual_widget,
    set_plot_style,
)
from sample_processing.model.baselines import apply_norm_scores, fit_norm_baselines
from sample_processing.model.preprocessing import clip_rms_spikes
from sample_processing.model.scenario_groups import add_scenario_group_labels

set_plot_style()

# 1. Dataset

The dataset contains vibration measurements from **29 independent sensor scenarios**. Each scenario provides two time windows:

| Split | File pattern | Label |
|-------|-------------|-------|
| `fit` | `vibe_data_fit_{1..29}.parquet` | Always `"normal"` — used as the healthy reference |
| `pred` | `vibe_data_pred_{1..29}.parquet` | `"normal"` or `"incident_N"` — evaluated against the model |

**Key columns:**
- `sampled_at` — UTC timestamp of each measurement
- `uptime` — `True` when the machine is running; `False` during idle/off periods
- `vel_rms_{x,y,z}` — velocity RMS (mm/s) on three axes
- `accel_rms_{x,y,z}` — acceleration RMS (g) on three axes

Incident windows are defined in `labels/incidents.yaml` and applied to `pred` rows only. Each `pred` row is labelled `"normal"` or `"incident_N"` depending on whether its timestamp falls within a known failure window.

In [ ]:
DATA_DIR = Path("../data")
LABELS_PATH = Path("../labels/incidents.yaml")

In [ ]:
# --- load incident windows ---------------------------------------------------
with open(LABELS_PATH) as f:
    _raw = yaml.safe_load(f)

# {scenario_id: [(start, end, incident_index), ...]}
_incidents: dict[int, list[tuple]] = {
    int(sid): [
        (pd.Timestamp(w["start"]), pd.Timestamp(w["end"]), i)
        for i, w in enumerate(windows, start=1)
    ]
    for sid, windows in _raw.items()
}


def _assign_label(ts: pd.Timestamp, windows: list[tuple]) -> str:
    """Return 'incident_N' if ts falls in the Nth window, else 'normal'."""
    for start, end, idx in windows:
        if start <= ts <= end:
            return f"incident_{idx}"
    return "normal"


# --- load fit files (always healthy) ----------------------------------------
fit_frames = []
for path in sorted(DATA_DIR.glob("vibe_data_fit_*.parquet"), key=lambda p: int(p.stem.split("_")[-1])):
    scenario_id = int(path.stem.split("_")[-1])
    df = pd.read_parquet(path)
    df.insert(0, "scenario_id", scenario_id)
    df.insert(1, "split", "fit")
    df.insert(2, "label", "normal")
    fit_frames.append(df)

# --- load pred files (label rows against incident windows) ------------------
pred_frames = []
for path in sorted(DATA_DIR.glob("vibe_data_pred_*.parquet"), key=lambda p: int(p.stem.split("_")[-1])):
    scenario_id = int(path.stem.split("_")[-1])
    df = pd.read_parquet(path)
    df.insert(0, "scenario_id", scenario_id)
    df.insert(1, "split", "pred")
    windows = _incidents.get(scenario_id, [])
    df.insert(2, "label", df["sampled_at"].apply(lambda ts: _assign_label(ts, windows)))
    pred_frames.append(df)

# --- combine -----------------------------------------------------------------
full_df = (
    pd.concat(fit_frames + pred_frames, ignore_index=True)
    .sort_values(["scenario_id", "sampled_at"])
    .reset_index(drop=True)
)

In [ ]:
summary = (
    full_df[['scenario_id', 'uptime', 'label']]
    .value_counts()
    .rename('count')
    .reset_index()
    .sort_values(['scenario_id', 'uptime', 'label'])
)


failure_labels = {'incident_1', 'incident_2'}
summary['failure'] = summary['label'].isin(failure_labels)

totals = (
    summary.groupby(['uptime', 'failure'])['count']
    .sum()
    .unstack(fill_value=0)
)

totals.columns = ['normal_count', 'failure_count']

pct = (
    totals.div(totals.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

pct.columns = ['normal_%', 'failure_%']

result = pd.concat([totals, pct], axis=1)
result

The distribution above reveals two important properties:

- **Incidents are rare** (< 4% of all rows) ? severe class imbalance that rules out supervised classification. Unsupervised baseline deviation is the chosen approach.
- **Incidents occur mostly during `uptime=True`**. However, anomalies are only measurable when the machine is operating. The `uptime=False` rows carry no incident signal and are excluded from healthy baseline fitting and scoring.

## Anomaly Detection Framework Context

With 29 isolated sensors and no peer / fleet context, this benchmark is strictly a **self-historical** anomaly-detection problem: learn a healthy baseline from the `fit` split and look for deviations in `pred`.

The preprocessing contract is intentionally simple:

- We keep **one signal-level transform only**: `clip_rms_spikes`.
- We fully trust **`uptime=True` as the operational gating signal**.
- We do **not** reinterpret low amplitude, level shifts, waiting periods, or temporary idle-looking behavior inside `uptime=True` as machine OFF.

That means a machine can stay operational even if the RMS level changes because of load, operating condition, or a waiting state. In this pipeline, those are modeled as part of the machine's active behavior as long as `uptime=True` stays true.

## Spike Clipping - Justification

`clip_rms_spikes(vel_threshold=100, accel_threshold=10)` removes sparse gross outliers before any baseline fitting. These thresholds sit orders of magnitude above the healthy operating band - velocity RMS is typically a few mm/s, acceleration RMS is typically sub-g - so any value above them is a transient artifact rather than mechanical signal. 

Alternatives considered, tested and rejected since they didn't work:

- **Butterworth / low-pass filtering.** Would smear fast-rising degradations and would require a justified cutoff that cannot be derived from this anonymous dataset. 
- **Regime inference from RMS amplitude.** Relabelling active rows as different regimes based on sudden magnitude shifts is not comprehensive enough based on the current available data.

In [ ]:
full_df = clip_rms_spikes(
    full_df,
    vel_threshold=100,
    accel_threshold=10,
)
fit_df = full_df[full_df["split"] == "fit"].copy()

In [ ]:
inspector = create_scenario_inspector(full_df, default_scenario=1)
display(inspector)

# 2. Preprocessing Pipeline

## Final Preprocessing Contract

The final preprocessing contract is intentionally narrow:

| Step | Purpose | Kept? |
|---|---|---|
| Spike clipping via `clip_rms_spikes` | Remove sparse gross outliers without redefining the signal shape | Yes |
| `uptime=True` gating | Define which rows count as active machine operation for fitting and scoring | Yes |
| Butterworth or similar frequency-based filtering | Smooth the RMS series using a cutoff / sampling-rate design | No  |
| Regime inference from RMS amplitude | Re-label active rows as ON/OFF based on sensor magnitude | No |

In [ ]:
_VEL_RAW = ["vel_rms_x", "vel_rms_y", "vel_rms_z"]
_ACCEL_RAW = ["accel_rms_x", "accel_rms_y", "accel_rms_z"]

full_df_final = full_df.copy()
full_df_final["global_mask_vel"] = ~full_df_final["uptime"].fillna(False).astype(bool)
full_df_final["global_mask_accel"] = ~full_df_final["uptime"].fillna(False).astype(bool)

print(f"global_mask_vel=True:   {full_df_final['global_mask_vel'].sum():,} rows ({full_df_final['global_mask_vel'].mean():.1%})")
print(f"global_mask_accel=True: {full_df_final['global_mask_accel'].sum():,} rows ({full_df_final['global_mask_accel'].mean():.1%})")
print(f"both usable: {(~full_df_final['global_mask_vel'] & ~full_df_final['global_mask_accel']).sum():,} rows")

In [ ]:
ui = create_rms_scenario_widget(
    full_df_final,
    default_scenarios=(1, 5, 12),
)
display(ui)

# 3. Healthy Baseline and Residual Calibration

Healthy baselines are fit directly on the raw RMS channels after spike clipping, using only rows where `uptime=True`. Each channel is standardized against its own fit-set mean and standard deviation, and the **residual** is `d_norm - threshold` - the distance, in fit-healthy standard deviations, above the threshold where scoring starts.

The residual is mapped to `[0, 1]` by a sigmoid:

```
score = 1 / (1 + exp(-alpha * (residual - beta)))
```

- **`beta`** is a residual-space offset; it shifts the sigmoid's midpoint to the right, which delays how quickly the score rises once the threshold is crossed. A larger beta means "ignore small residuals".
- **`alpha`** is the sigmoid slope; it controls how abruptly the score jumps from near-0 to near-1 across beta. A larger alpha means a harder decision boundary.

Calibration is done in residual space with two anchors per modality, `(r1, p1)` and `(r2, p2)`, where `r` is a residual value and `p` is the desired sigmoid score at that residual. With 2 parameters `alpha` and `beta`, we use known `(r1, p1)` and `(r2, p2)` to solve a system of 2 equations and 2 variables. The widget solves `alpha` and `beta` from those anchor pairs and overlays both the anchors and the resulting sigmoid curve on the pooled residual histograms, so the choice is made by visual fit rather than by guessing hyperparameters. 

The global anchors changes depending on the groupings (see `src/sample_processing/hyperparameters/norm_model_hyperparams.yaml`). The per-group values in the next notebook are derived from this widget too, once the scenarios are grouped below in Section 4.

In [ ]:
fit_df = full_df_final[full_df_final["split"] == "fit"].copy()

norm_baselines = fit_norm_baselines(
    df=fit_df,
    scaler="standard",
    vel_cols=_VEL_RAW,
    accel_cols=_ACCEL_RAW,
    vel_mask_col="global_mask_vel",
    accel_mask_col="global_mask_accel",
)

full_scored_df = apply_norm_scores(
    df=full_df_final,
    baselines=norm_baselines,
    vel_cols=_VEL_RAW,
    accel_cols=_ACCEL_RAW,
    vel_mask_col="global_mask_vel",
    accel_mask_col="global_mask_accel",
)

In [ ]:
ui, get_sigmoid_params = create_sigmoid_global_residual_widget(
    full_scored_df,
    vel_cols=_VEL_RAW,
    accel_cols=_ACCEL_RAW,
    default_num_bins=100,
    threshold_vel=3.0,
    threshold_accel=3.0,
    alpha_vel=0.74,
    beta_vel=1.86,
    alpha_accel=0.74,
    beta_accel=1.86,
)
display(ui)

In [ ]:
sigmoid_params = get_sigmoid_params()
sigmoid_params

In [ ]:
full_scored_df.columns

# 4. Manual Grouping From `.pred` Behavior

The scenarios are grouped here using the **observed `.pred` degradation shape** seen in the residual RMS views. This is an **offline surrogate for missing asset metadata**: in a real production setting these group labels would come from machine metadata (asset family, machine type, operating context), not from retrospective signal inspection.

**Methodology.** For each scenario, we compared the residual trajectory on the `pred` split against its healthy baseline using the scoring and distribution widgets. Scenarios that shared the same failure archetype were assigned to the same group. Four archetypes emerged (`src\analysis\scenario_groups.py`):

| Group | Archetype | Scenarios | Count |
|---|---|---|---|
| **Group 1** | Large single spike on top of a flat baseline | 1, 2, 4, 7, 9, 11, 17 | 7 |
| **Group 2** | Sudden sustained trend increase | 3, 6, 10, 15, 16, 18, 20, 21, 22, 24, 26, 27 | 12 |
| **Group 3** | Segments with sharp increases and back to normal | 5, 8, 19, 25, 28, 29 | 6 |
| **Group 4** | Many ups and downs with a localized degradation band | 12, 13, 14, 23 | 4 |

These archetypes motivate per-group sigmoid calibration in `02_model_debugging.ipynb`: a single global `(alpha, beta, threshold, window_top_k)` cannot simultaneously catch an isolated spike (needs a sharp, sensitive sigmoid) and a gradual trend drift (needs a gentler slope and higher threshold) without accepting false positives somewhere. I tried that and didn't work. 

The archetype is from my analysis, but I had to reassign scenarios if another group improved the performance metrics. 

The per-group values live in `src/sample_processing/hyperparameters/norm_model_hyperparams.yaml` and are validated with the group-distribution widget below.

In [ ]:
full_df = add_scenario_group_labels(full_df)
full_df_final = add_scenario_group_labels(full_df_final)
full_scored_df = add_scenario_group_labels(full_scored_df)

full_scored_df[["scenario_id", "scenario_group", "scenario_group_label"]].drop_duplicates().sort_values("scenario_id").reset_index(drop=True)

In [ ]:

ui = create_group_distribution_widget(
    full_scored_df,
    vel_cols=_VEL_RAW,
    accel_cols=_ACCEL_RAW,
    threshold_vel=sigmoid_params["threshold_vel"],
    threshold_accel=sigmoid_params["threshold_accel"],
)
display(ui)


### Final Modeling Assumptions and Handoff

- **Only one signal-level preprocessing step is used:** spike clipping.
- **`uptime=True` is trusted as machine operating**, even when the active machine goes through level shifts, load changes, or waiting periods with low vibration amplitude.
- **Healthy baselines and residual scoring** are computed directly from the raw RMS channels after that gating step.
- **Scenarios are partitioned into four groups** as a metadata surrogate, and each group carries its own `(alpha, beta, threshold, window_top_k, fusion_threshold)`.

Notebook `02_model_debugging.ipynb` loads those per-group hyperparameters, fits all 29 scenarios, replays the production API path in `execution_mode='api_exact'`, and validates the alert engine against the labeled incident windows. The group archetypes above are what make per-group tuning worth the added complexity - they are the premise tested there.